# Clothes segmentation: two fine-tuning experiments on the final model

Both start from the current best model, `final_segformer_b3_e30/best.pt`, which is only read and never changed. Both are judged on the validation split by rules fixed in PLAN.md, section 26.

- **Cell 6, rank 2:** a short fine-tune with weight averaging. About 2 hours.
- **Cell 7, rank 3:** a fine-tune at a larger input, 768×512. About 1.5 hours.

Use the **L4 GPU**. Run the cells from top to bottom; cell 3 asks for `clothseg_code.zip` once. If the session ends, run cells 1 to 5 and the interrupted cell again: it continues from the last finished epoch.

## 1. Settings

In [ ]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements and prepare the dataset

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 5. Helper for comparing a new model with the current best one

Run this once before cells 6 and 7. It evaluates a checkpoint on the validation split at the original image size with flip averaging, and prints it next to the current best model.

In [ ]:
import json, subprocess, sys
import pandas as pd

BASE_RUN = "final_segformer_b3_e30"
BASE_CKPT = f"{DRIVE_ROOT}/runs/{BASE_RUN}/checkpoints/best.pt"
CORE = ["upper-clothes", "pants", "skirt", "dress"]
WATCH = CORE + ["belt", "scarf", "sunglasses", "shoes"]
assert os.path.exists(BASE_CKPT), f"The current best model is missing: {BASE_CKPT}"

def val_eval(checkpoint, tag):
    out = f"{DRIVE_ROOT}/reports/eval_{tag}_val_flip"
    print(f"evaluating {tag} on validation ...", flush=True)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", "--checkpoint", checkpoint, "--split", "val", "--out-dir", out, "--flip-tta"],
                       capture_output=True, text=True, cwd=CODE_DIR)
    assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
    return json.load(open(f"{out}/metrics.json")), pd.read_csv(f"{out}/per_class.csv").set_index("class")["iou"]

def compare(checkpoint, tag):
    """Print the new model next to the current best one and say whether it meets the rule fixed in PLAN.md, section 26."""
    assert os.path.exists(checkpoint), f"No checkpoint at {checkpoint}. Did the training cell finish?"
    base_m, base_pc = val_eval(BASE_CKPT, BASE_RUN)
    new_m, new_pc = val_eval(checkpoint, tag)
    rows = {k: (base_m[k], new_m[k]) for k in ("miou", "garment_miou", "core_garment_miou", "clothing_binary_iou", "boundary_miou")}
    rows.update({c: (base_pc[c], new_pc[c]) for c in WATCH})
    df = pd.DataFrame(rows, index=["current best", tag]).T
    df["change"] = df[tag] - df["current best"]
    print()
    print(df.round(4).to_string())
    gain = new_m["garment_miou"] - base_m["garment_miou"]
    worst_core = min(new_pc[c] - base_pc[c] for c in CORE)
    ok = gain >= 0.005 and worst_core >= -0.01
    print()
    print(f"garment mIoU change {100 * gain:+.2f} points; largest drop on a core garment {100 * worst_core:+.2f} points -> "
          + ("MEETS the rule" if ok else "does NOT meet the rule") + " (at least +0.50, and no core garment below -1.00)")

print("helper ready")

## 6. Rank 2: short fine-tune with weight averaging

Continues from the current best model in 3 cycles of 3 epochs. Each cycle starts at a quarter of the original learning rate and decays; the weights at the end of each cycle are averaged together with the starting weights. About 2 hours on the L4. The current best model is only read, never changed. If the session ends, run cells 1 to 5 and this cell again.

In [ ]:
SWA_RUN = "swa_segformer_b3"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 5 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/finetune.py --drive-root "{DRIVE_ROOT}" --init-checkpoint "{BASE_CKPT}" --run-name {SWA_RUN} --cycles 3 --epochs-per-cycle 3 --time-budget-min {budget}

compare(f"{DRIVE_ROOT}/runs/{SWA_RUN}/checkpoints/best.pt", SWA_RUN)

## 7. Rank 3: fine-tune at a larger input (768×512)

It first builds the dataset at 768×512 (several minutes, once), then fine-tunes the current best model for 3 epochs at batch size 8. About 1.5 hours on the L4.

In [ ]:
BIG_RUN = "big_segformer_b3_768"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 5 and this cell again.")

os.chdir(CODE_DIR)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height 768 --width 512 --workers 8
budget = int(minutes_left())
!python scripts/finetune.py --drive-root "{DRIVE_ROOT}" --init-checkpoint "{BASE_CKPT}" --run-name {BIG_RUN} --height 768 --width 512 --batch-size 8 --cycles 1 --epochs-per-cycle 3 --no-average --time-budget-min {budget}

compare(f"{DRIVE_ROOT}/runs/{BIG_RUN}/checkpoints/best.pt", BIG_RUN)

## 8. Do two models have different strengths? (ensemble test)

No training. It averages the predictions of the final model with those of other trained models and evaluates on the validation split. A clear gain means the models are right in different places, which is the condition for a merged architecture to be worth building. About 25 minutes. Only cells 1 to 4 are needed before it.

In [ ]:
import json, os, subprocess, sys
import pandas as pd

os.chdir(CODE_DIR)
R = f"{DRIVE_ROOT}/runs"
CKPT = {
    "final B3": f"{R}/final_segformer_b3_e30/checkpoints/best.pt",
    "UPerNet": f"{R}/r1_upernet_convnext_tiny/checkpoints/best.pt",
    "B4": f"{R}/r1_segformer_b4/checkpoints/best.pt",
    "B3 lr x2": f"{R}/abl_segformer_b3_lr_double/checkpoints/best.pt",
}
COMBOS = [["final B3"], ["UPerNet"], ["B4"], ["B3 lr x2"],
          ["final B3", "UPerNet"], ["final B3", "B4"], ["final B3", "B3 lr x2"], ["final B3", "UPerNet", "B4"]]
WATCH = ["upper-clothes", "pants", "skirt", "dress", "belt", "scarf", "sunglasses"]
missing = [k for k, v in CKPT.items() if not os.path.exists(v)]
COMBOS = [c for c in COMBOS if not set(c) & set(missing)]
if missing:
    print("not found on Drive, left out:", missing)

rows = {}
for combo in COMBOS:
    name = " + ".join(combo)
    out = f"{DRIVE_ROOT}/reports/ens_" + "_".join(k.replace(" ", "") for k in combo)
    print("evaluating", name, "...", flush=True)
    cmd = [sys.executable, "scripts/evaluate.py", "--checkpoint", CKPT[combo[0]], "--split", "val", "--out-dir", out, "--flip-tta"]
    if len(combo) > 1:
        cmd += ["--ensemble-with"] + [CKPT[k] for k in combo[1:]]
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, "The evaluation failed. Send me this message: " + (r.stdout + r.stderr)[-3000:]
    m = json.load(open(f"{out}/metrics.json"))
    pc = pd.read_csv(f"{out}/per_class.csv").set_index("class")["iou"]
    rows[name] = {"mIoU": m["miou"], "garment": m["garment_miou"], "core": m["core_garment_miou"], "boundary": m["boundary_miou"],
                  **{c: pc[c] for c in WATCH}, "ms/img": m["ms_per_image"]}

df = pd.DataFrame(rows).T
print()
print("VALIDATION, original resolution, flip averaging")
print(df.round(4).to_string())
base = df.loc["final B3"]
print()
for name in df.index:
    if "+" in name:
        gain = df.loc[name, "mIoU"] - base["mIoU"]
        print(f"{name}: mIoU {100 * gain:+.2f} points, garment mIoU {100 * (df.loc[name, 'garment'] - base['garment']):+.2f} points over the final model alone -> "
              + ("CLEAR gain" if gain >= 0.005 else "no clear gain") + " (threshold +0.50)")

## 9. Merged architecture: SegFormer-B3 encoder + ConvNeXt-Tiny encoder with a fusion decoder

One network with two encoders. Both read the photo; their features are joined at each of the four scales; a new decoder predicts the labels. Trained for 12 epochs with the same data, seed and settings as the single models from the comparison, then printed next to them. About 3 hours on the L4. Only cells 1 to 4 are needed before it. If the session ends, run cells 1 to 4 and this cell again.

In [ ]:
import os
import pandas as pd

DUAL_RUN = "abl_dual_b3_convnext_tiny"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {DUAL_RUN} --model dual_b3_convnext_tiny --height {HEIGHT} --width {WIDTH} --epochs 12 --time-budget-min {budget}

COLS = ["garment_miou", "miou", "iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress", "iou_belt", "iou_scarf", "iou_sunglasses", "val_loss", "train_min"]
CORE = ["iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress"]
RUNS = {"SegFormer-B3": "r1_segformer_b3", "UPerNet+ConvNeXt-T": "r1_upernet_convnext_tiny", "merged": DUAL_RUN}
logs = {k: pd.read_csv(f"{DRIVE_ROOT}/runs/{v}/val_log.csv").set_index("epoch") for k, v in RUNS.items()
        if os.path.exists(f"{DRIVE_ROOT}/runs/{v}/val_log.csv")}
assert "merged" in logs, "The merged model has not finished an epoch yet. Read the training output above for the reason."
print()
print("garment mIoU per epoch")
print(pd.DataFrame({k: v["garment_miou"] for k, v in logs.items()}).round(4).to_string())
done = len(logs["merged"])
if done < 12:
    print()
    print(f"NOT FINISHED: {done} of 12 epochs. Run this cell again to continue.")
else:
    print()
    print("validation at epoch 12")
    print(pd.DataFrame({k: v.loc[12, COLS] for k, v in logs.items()}).round(4).to_string())
    base, new = logs["SegFormer-B3"].loc[12], logs["merged"].loc[12]
    gain = new["garment_miou"] - base["garment_miou"]
    worst = min(new[c] - base[c] for c in CORE)
    ok = gain >= 0.005 and worst >= -0.01
    print()
    print(f"merged against SegFormer-B3: garment mIoU {100 * gain:+.2f} points; largest drop on a core garment {100 * worst:+.2f} points -> "
          + ("MEETS the rule" if ok else "does NOT meet the rule") + " (at least +0.50, and no core garment below -1.00)")

## What to send back

Everything cells 6, 7, 8 and 9 print. The table and the last lines of each are the important part.